# TurAIsta Intent Model: Fine-Tuning and GGUF Export

This notebook trains the on-device model that reads a guest message and returns its intent and slots as JSON, for example:

`{"intent": "BOOKING", "slots": {"party_size": "4", "date": "Saturday"}}`

Intents are BOOKING, PRICE, DIRECTIONS, DIETARY_SAFETY, and OTHER. The base model is Qwen2.5-0.5B-Instruct (Apache 2.0, commercial use allowed), fine-tuned with LoRA and exported to GGUF for the Flutter app.

Changes from v1, based on its results (98.1 percent on template test, 81.8 percent on wild test):

* The safety detector now covers Tagalog and Taglish. v1 missed "Bawal po ako sa gatas" because the detector only knew en, ko, ja, and zh.
* Casual templates and chat-style augmentation (lowercase, no punctuation, abbreviations such as "hm", "pls", "san") were added, because v1 labeled casual price and directions questions as OTHER.
* The wild test set is unchanged, so v1 and v2 can be compared fairly on it. The template test set is larger in v2 because it now includes casual templates.

Earlier design decisions still apply:

* Training data is farm-tour templates in en, tl, ko, ja, and zh, because no public dataset covers these intents.
* The split is by template, so test sentences come from templates the model never saw.
* A rule-based safety detector runs alongside the model. Any detector hit forces DIETARY_SAFETY, and the evaluation reports recall with and without this override.

## Setup Checklist

1. Session options: Accelerator GPU T4 x1, Internet On.
2. No Hugging Face token is required for this notebook.
3. Run All. Expected runtime is 25 to 45 minutes.

## 1. Install

In [1]:
!pip install -q "peft>=0.11" "accelerate>=0.30" gguf sentencepiece
# Kaggle ships torchao 0.10, which recent peft rejects. LoRA does not need it.
!pip uninstall -y -q torchao

## 2. Project Modules

The safety bank and the templates are written to disk so the notebook matches the repository code exactly.

In [2]:
%%writefile bank.py
"""TurAIsta safety phrase bank and rule-based DIETARY_SAFETY detector.

Why this exists: the 2026-10-04 evaluation showed NLLB-200 mistranslating
"allergic to peanuts" as seeds (tl), apples (ceb) and vegetables (ilo). Safety
messages therefore never go through free translation. Instead:
  1. detect() scans the guest's message for allergen and risk keywords in
     en, tl (incl. Taglish), ko, ja, zh. It is deliberately high-recall and runs BEFORE the intent model.
  2. Any hit forces intent DIETARY_SAFETY, whatever the model says.
  3. card() renders a fixed host-language card from verified terms only.

Every host-language term is a DRAFT until a native speaker sets reviewed=True.
card() refuses to show an unreviewed term without a visible warning.
"""
from __future__ import annotations

import re
import unicodedata

# confidence: "high" = standard everyday word; "check" = plausible, needs a native speaker.
# Guest-side keywords are matched as substrings (CJK) or word prefixes (en).
ITEMS = {
    # ---- allergens
    "peanut": {"kind": "allergen",
               "guest": {"tl": ['mani'], "en": ["peanut", "groundnut"], "ko": ["땅콩"], "ja": ["ピーナッツ", "落花生", "ピーナツ"], "zh": ["花生"]},
               "host": {"tl": ("mani", "high"), "ceb": ("mani", "high"), "ilo": ("mani", "high")}},
    "tree_nut": {"kind": "allergen",
                 "guest": {"tl": ['kasoy', 'almendras'], "en": ["tree nut", "nuts", "almond", "cashew", "walnut"], "ko": ["견과", "아몬드", "호두", "캐슈"],
                           "ja": ["ナッツ", "アーモンド", "くるみ", "クルミ", "カシュー"], "zh": ["坚果", "堅果", "杏仁", "核桃", "腰果"]},
                 "host": {"tl": ("mga nuwes (kasoy, almendras)", "check"), "ceb": ("mga nuts (kasoy, almendras)", "check"),
                          "ilo": ("mga nuts (kasoy)", "check")}},
    "shellfish": {"kind": "allergen",
                  "guest": {"tl": ['hipon', 'alimango', 'alimasag', 'talaba', 'tahong', 'pusit', 'shellfish', 'seafood'], "en": ["shellfish", "shrimp", "prawn", "crab", "lobster", "crustacean"], "ko": ["갑각류", "새우", "게", "조개", "랍스터"],
                            "ja": ["甲殻類", "えび", "エビ", "海老", "かに", "カニ", "蟹", "貝"], "zh": ["甲壳", "甲殼", "虾", "蝦", "蟹", "贝", "貝"]},
                  "host": {"tl": ("hipon, alimango, alimasag, at iba pang shellfish", "high"),
                           "ceb": ("pasayan, kasag, alimango, kinhason", "check"),
                           "ilo": ("padaw, kappi, ken dadduma pay a shellfish", "check")}},
    "fish": {"kind": "allergen",
             "guest": {"tl": ['isda'], "en": ["fish"], "ko": ["생선", "물고기"], "ja": ["魚", "さかな"], "zh": ["鱼", "魚"]},
             "host": {"tl": ("isda", "high"), "ceb": ("isda", "high"), "ilo": ("ikan", "check")}},
    "egg": {"kind": "allergen",
            "guest": {"tl": ['itlog'], "en": ["egg"], "ko": ["달걀", "계란"], "ja": ["卵", "たまご", "玉子"], "zh": ["鸡蛋", "雞蛋", "蛋"]},
            "host": {"tl": ("itlog", "high"), "ceb": ("itlog", "high"), "ilo": ("itlog", "high")}},
    "milk": {"kind": "allergen",
             "guest": {"tl": ['gatas', 'keso', 'dairy'], "en": ["milk", "dairy", "lactose", "cheese", "butter"], "ko": ["우유", "유제품", "유당", "치즈", "버터"],
                       "ja": ["牛乳", "乳製品", "乳糖", "チーズ", "バター", "ミルク"], "zh": ["牛奶", "奶制品", "乳制品", "乳糖", "奶酪", "黄油"]},
             "host": {"tl": ("gatas at mga produktong gawa sa gatas", "high"), "ceb": ("gatas ug mga produkto sa gatas", "check"),
                      "ilo": ("gatas ken dagiti produkto ti gatas", "check")}},
    "wheat_gluten": {"kind": "allergen",
                     "guest": {"tl": ['gluten', 'trigo', 'harina'], "en": ["wheat", "gluten", "celiac", "coeliac"], "ko": ["밀", "글루텐"], "ja": ["小麦", "グルテン"], "zh": ["小麦", "麸质", "麩質", "面筋"]},
                     "host": {"tl": ("trigo / harina (gluten)", "check"), "ceb": ("trigo / harina (gluten)", "check"),
                              "ilo": ("trigo / harina (gluten)", "check")}},
    "soy": {"kind": "allergen",
            "guest": {"tl": ['toyo', 'tokwa', 'soya'], "en": ["soy", "soya", "tofu"], "ko": ["대두", "콩", "두부"], "ja": ["大豆", "豆腐", "醤油"], "zh": ["大豆", "黄豆", "豆腐", "酱油"]},
            "host": {"tl": ("soya / toyo / tokwa", "check"), "ceb": ("soya / toyo / tokwa", "check"), "ilo": ("soya / toyo / tokwa", "check")}},
    "sesame": {"kind": "allergen",
               "guest": {"tl": ['linga'], "en": ["sesame"], "ko": ["참깨", "깨"], "ja": ["ごま", "ゴマ", "胡麻"], "zh": ["芝麻"]},
               "host": {"tl": ("linga", "high"), "ceb": ("linga", "check"), "ilo": ("linga", "check")}},
    # ---- diets (the card asks the host to confirm ingredients, it never promises safety)
    "vegetarian": {"kind": "diet",
                   "guest": {"tl": ['vegetarian', 'walang karne', 'gulay lang'], "en": ["vegetarian", "no meat"], "ko": ["채식", "베지테리언"], "ja": ["ベジタリアン", "菜食"], "zh": ["素食", "吃素"]},
                   "host": {"tl": ("walang karne (vegetarian)", "high"), "ceb": ("walay karne (vegetarian)", "check"),
                            "ilo": ("awan ti karne (vegetarian)", "check")}},
    "vegan": {"kind": "diet",
              "guest": {"tl": ['vegan'], "en": ["vegan"], "ko": ["비건", "완전 채식"], "ja": ["ヴィーガン", "ビーガン"], "zh": ["纯素", "純素"]},
              "host": {"tl": ("walang anumang galing sa hayop: karne, isda, itlog, gatas (vegan)", "high"),
                       "ceb": ("walay bisan unsa gikan sa hayop (vegan)", "check"), "ilo": ("awan ti aniaman a naggapu iti animal (vegan)", "check")}},
    "halal": {"kind": "diet",
              "guest": {"tl": ['halal', 'muslim'], "en": ["halal", "muslim"], "ko": ["할랄"], "ja": ["ハラール", "ハラル"], "zh": ["清真"]},
              "host": {"tl": ("halal: walang baboy at alak", "high"), "ceb": ("halal: walay baboy ug alak", "check"),
                       "ilo": ("halal: awan ti baboy ken arak", "check")}},
    "no_pork": {"kind": "diet",
                "guest": {"tl": ['baboy'], "en": ["pork", "pig"], "ko": ["돼지고기"], "ja": ["豚肉", "ポーク"], "zh": ["猪肉", "豬肉"]},
                "host": {"tl": ("baboy", "high"), "ceb": ("baboy", "high"), "ilo": ("baboy", "high")}},
    "no_alcohol": {"kind": "diet",
                   "guest": {"tl": ['alak', 'beer', 'inuman'], "en": ["alcohol", "no wine", "no beer"], "ko": ["술", "알코올"], "ja": ["お酒", "アルコール"], "zh": ["酒精", "不喝酒"]},
                   "host": {"tl": ("alak", "high"), "ceb": ("alak / ilimnon", "check"), "ilo": ("arak", "check")}},
    # ---- health, children, access, weather risk
    "allergy_general": {"kind": "health",
                        "guest": {"tl": ['allerg', 'bawal ako', 'bawal po ako', 'bawal kami', 'bawal sa', 'di pwede kumain', 'hindi pwede kumain', 'hindi makakain', 'di makakain'], "en": ["allerg", "anaphyla", "epipen"], "ko": ["알레르기", "알러지", "에피펜"], "ja": ["アレルギー", "エピペン"], "zh": ["过敏", "過敏"]},
                        "host": {"tl": ("may allergy (maaaring delikado)", "high"), "ceb": ("adunay alerdyi (mahimong delikado)", "check"),
                                 "ilo": ("adda alerdyi (mabalin a peligro)", "check")}},
    "medical": {"kind": "health",
                "guest": {"tl": ['buntis', 'diabetic', 'hika', 'sakit', 'gamot', 'hilo', 'nahihilo', 'ospital', 'doktor', 'high blood', 'altapresyon'], "en": ["diabet", "asthma", "heart", "medicine", "pregnan", "sick", "hospital", "doctor"],
                          "ko": ["당뇨", "천식", "심장", "약", "임신", "아파", "병원", "의사"],
                          "ja": ["糖尿", "喘息", "心臓", "薬", "妊娠", "具合", "病院", "医者"],
                          "zh": ["糖尿", "哮喘", "心脏", "心臟", "药", "藥", "怀孕", "懷孕", "生病", "医院", "醫院", "医生"]},
                "host": {"tl": ("may kondisyong pangkalusugan", "high"), "ceb": ("adunay kondisyon sa panglawas", "check"),
                         "ilo": ("adda kondision ti salun-at", "check")}},
    "child": {"kind": "child",
              "guest": {"tl": ['anak', 'bata', 'bunso', 'baby', 'sanggol', 'paslit'], "en": ["child", "kid", "son", "daughter", "baby", "toddler", "infant"], "ko": ["아이", "아들", "딸", "아기", "어린이"],
                        "ja": ["子供", "子ども", "息子", "娘", "赤ちゃん"], "zh": ["孩子", "儿子", "兒子", "女儿", "女兒", "婴儿", "嬰兒", "小孩"]},
              "host": {"tl": ("may kasamang bata", "high"), "ceb": ("adunay kauban nga bata", "check"), "ilo": ("adda kadua nga ubing", "check")}},
    "accessibility": {"kind": "access",
                      "guest": {"tl": ['wheelchair', 'pilay', 'tungkod', 'senior', 'matanda', 'lolo', 'lola', 'pwd'], "en": ["wheelchair", "disab", "walking stick", "cane", "elderly", "mobility"], "ko": ["휠체어", "장애", "지팡이", "노인"],
                                "ja": ["車椅子", "車いす", "障害", "杖", "高齢"], "zh": ["轮椅", "輪椅", "残疾", "殘疾", "拐杖", "老人"]},
                      "host": {"tl": ("kailangan ng tulong sa paglalakad / wheelchair", "high"),
                               "ceb": ("nagkinahanglan og tabang sa paglakaw / wheelchair", "check"),
                               "ilo": ("masapul ti tulong iti panagnagna / wheelchair", "check")}},
    "weather_risk": {"kind": "weather",
                     "guest": {"tl": ['ulan', 'umuulan', 'bagyo', 'baha', 'madulas', 'init', 'mainit', 'landslide', 'kidlat'], "en": ["rain", "typhoon", "storm", "flood", "slippery", "heat", "landslide"],
                               "ko": ["비", "태풍", "폭풍", "홍수", "미끄", "더위", "산사태"],
                               "ja": ["雨", "台風", "嵐", "洪水", "滑り", "暑", "土砂"],
                               "zh": ["雨", "台风", "颱風", "暴风", "洪水", "滑", "炎热", "山体滑坡"]},
                     "host": {"tl": ("nagtatanong tungkol sa panganib dahil sa panahon", "high"),
                              "ceb": ("nangutana bahin sa peligro tungod sa panahon", "check"),
                              "ilo": ("agsaludsod maipanggep iti peligro gapu iti panawen", "check")}},
}
# Words whose very short CJK keywords cause false positives; matched only as whole tokens.
SHORT_KO = {"게", "밀", "콩", "깨", "술", "약", "비", "딸", "아이", "아기", "아들"}
KO_PARTICLES = ("", "이", "가", "은", "는", "을", "를", "에", "도", "와", "과", "랑", "의", "들", "들이", "들은", "들도", "에게", "이가", "이는", "이에요", "예요")

CARD_TITLE = {"tl": "PAALALA SA KALIGTASAN", "ceb": "PAHINUMDOM SA KALUWASAN", "ilo": "PALAGIP MAIPANGGEP ITI KINATALGED"}
CARD_ASK = {"tl": "Pakisuri ang lahat ng sangkap bago ihain. Kung hindi sigurado, huwag ihain.",
            "ceb": "Palihug susiha ang tanang sangkap sa dili pa idalit. Kung dili sigurado, ayaw idalit.",
            "ilo": "Pangngaasiyo ta kitaenyo amin a ramen sakbay nga idasar. No saan a siguradokayo, dikay idasar."}
REVIEWED: dict[tuple[str, str], bool] = {}   # (item, host_lang) -> True once a native speaker approves


def _norm(t: str) -> str:
    return unicodedata.normalize("NFKC", t).lower()


def detect(text: str) -> list[str]:
    """Return item keys whose guest keywords appear in text, in any guest language."""
    t = _norm(text)
    tokens = set(re.findall(r"[\w']+", t))
    hits = []
    for key, item in ITEMS.items():
        for lang, words in item["guest"].items():
            for w in words:
                w = _norm(w)
                if lang == "en" or (lang == "tl" and w == "allerg"):
                    ok = re.search(r"\b" + re.escape(w), t) is not None
                elif lang == "tl":  # whole words, so "bata" does not match "Batangas"
                    ok = re.search(r"\b" + re.escape(w) + r"\b", t) is not None
                elif lang == "ko" and w in SHORT_KO:
                    ok = any(tok == w + p for tok in tokens for p in KO_PARTICLES)
                else:
                    ok = w in t
                if ok:
                    hits.append(key)
                    break
            if hits and hits[-1] == key:
                break
    return hits


def card(hits: list[str], host_lang: str) -> str:
    """Fixed host-language safety card. Unreviewed terms are visibly marked."""
    lines = [CARD_TITLE[host_lang]]
    for key in hits:
        term, conf = ITEMS[key]["host"][host_lang]
        flag = "" if REVIEWED.get((key, host_lang)) else " [draft]"
        lines.append(f"- {term.upper()}{flag}")
    lines.append(CARD_ASK[host_lang])
    return "\n".join(lines)


def route(text: str, model_intent: str) -> str:
    """Safety override: any detector hit forces DIETARY_SAFETY."""
    return "DIETARY_SAFETY" if detect(text) else model_intent


def review_rows():
    """Rows for the native-speaker review sheet."""
    for key, item in ITEMS.items():
        for h, (term, conf) in item["host"].items():
            yield {"item": key, "kind": item["kind"], "host_lang": h, "draft_term": term,
                   "confidence": conf, "english_meaning": key.replace("_", " "),
                   "approved (yes/no)": "", "correction": "", "reviewer": ""}


Overwriting bank.py


In [3]:
%%writefile templates.py
"""Farm-tour message templates for the TurAIsta intent model.

Intents: BOOKING, PRICE, DIRECTIONS, DIETARY_SAFETY, OTHER.
Slots: party_size, date, time, place, item, allergen.

REVIEW STATUS: en written by a fluent speaker; tl, ko, ja, zh need native-speaker
review before any result is reported as final. Each template has an id so the
train/test split is made by template, never by filled sentence (no leakage).
"""
NEEDS_REVIEW = {"tl", "ko", "ja", "zh"}

FILL = {
 "en": {"n": ["2", "3", "4", "five", "6"], "day": ["Saturday", "Sunday", "tomorrow", "next Friday", "March 12"],
        "time": ["9am", "10:30", "early morning", "2pm"], "place": ["the bus terminal", "the town plaza", "the highway", "the airport", "Baguio"],
        "item": ["the farm tour", "lunch", "strawberry picking", "the coffee tasting", "the overnight stay"],
        "allergen": ["peanuts", "shrimp", "eggs", "milk", "shellfish", "gluten"]},
 "tl": {"n": ["2", "3", "apat", "lima", "6"], "day": ["Sabado", "Linggo", "bukas", "sa Biyernes", "Marso 12"],
        "time": ["9 ng umaga", "10:30", "maaga", "2 ng hapon"], "place": ["terminal ng bus", "plaza", "highway", "airport", "Baguio"],
        "item": ["farm tour", "tanghalian", "strawberry picking", "coffee tasting", "overnight stay"],
        "allergen": ["mani", "hipon", "itlog", "gatas", "shellfish", "gluten"]},
 "ko": {"n": ["2명", "3명", "4명", "다섯 명", "6명"], "day": ["토요일", "일요일", "내일", "다음 주 금요일", "3월 12일"],
        "time": ["오전 9시", "10시 반", "아침 일찍", "오후 2시"], "place": ["버스 터미널", "시내 광장", "고속도로", "공항", "바기오"],
        "item": ["농장 투어", "점심", "딸기 따기", "커피 시음", "1박"],
        "allergen": ["땅콩", "새우", "달걀", "우유", "갑각류", "밀"]},
 "ja": {"n": ["2人", "3人", "4人", "5人", "6人"], "day": ["土曜日", "日曜日", "明日", "来週の金曜日", "3月12日"],
        "time": ["午前9時", "10時半", "朝早く", "午後2時"], "place": ["バスターミナル", "町の広場", "高速道路", "空港", "バギオ"],
        "item": ["農場ツアー", "ランチ", "いちご狩り", "コーヒーの試飲", "1泊"],
        "allergen": ["ピーナッツ", "えび", "卵", "牛乳", "甲殻類", "小麦"]},
 "zh": {"n": ["两个人", "三个人", "四个人", "五个人", "六个人"], "day": ["星期六", "星期天", "明天", "下周五", "3月12日"],
        "time": ["上午九点", "十点半", "一大早", "下午两点"], "place": ["汽车站", "镇广场", "高速公路", "机场", "碧瑶"],
        "item": ["农场游", "午餐", "摘草莓", "咖啡品尝", "住一晚"],
        "allergen": ["花生", "虾", "鸡蛋", "牛奶", "甲壳类", "小麦"]},
}
SLOT_OF = {"n": "party_size", "day": "date", "time": "time", "place": "place", "item": "item", "allergen": "allergen"}

T = {
 "en": {
  "BOOKING": ["Can we book {item} for {n} people on {day}?", "I'd like to reserve a slot for {day} at {time}.",
              "Is there still space for {n} on {day}?", "Please book us for {item}.", "Can I change our booking to {day}?",
              "We want to cancel our reservation for {day}.", "Are you open on {day}? We'd like to come.",
              "Do I need to reserve {item} in advance?", "Reserve for {n} please, arriving around {time}.",
              "Can we join {item} on {day} morning?"],
  "PRICE": ["How much is {item} per person?", "What's the price for {n} people?", "Is there an entrance fee?",
            "How much for {item}?", "Do you accept GCash or cash only?", "Is {item} included in the price?",
            "Is there a student discount?", "Is there a group rate for {n}?", "How much is the deposit?", "Can I pay when we arrive?"],
  "DIRECTIONS": ["How do we get to your farm from {place}?", "Where exactly is the farm?", "Can you send a map pin?",
                 "Is there parking for a van?", "Which jeepney goes to your farm from {place}?", "How far is it from {place}?",
                 "We're lost near {place}, which way?", "Can a tricycle reach the farm?", "What landmark should we look for?",
                 "How long is the drive from {place}?"],
  "DIETARY_SAFETY": ["My son is allergic to {allergen}. Is {item} safe for him?", "Does the food contain {allergen}?",
                     "I can't eat {allergen}, please tell the cook.", "We are vegetarian, is there food for us?",
                     "Is the lunch halal?", "My mother uses a wheelchair. Can she join {item}?",
                     "Is the trail safe if it rains?", "Is there shade? My daughter gets sick in the heat.",
                     "Are there bees or animals that bite? My kid is 3.", "I'm pregnant, is {item} too tiring?"],
  "OTHER": ["Hello!", "Thank you so much, see you!", "What animals do you have on the farm?", "Can we take photos?",
            "Is there wifi?", "What's the best season to visit?", "We had a great time, salamat!", "Do you sell souvenirs?",
            "Good morning!", "What crops do you grow?"]},
 "tl": {
  "BOOKING": ["Pwede po ba mag-book ng {item} para sa {n} tao sa {day}?", "Gusto ko pong magpa-reserve sa {day} ng {time}.",
              "May slot pa po ba para sa {n} sa {day}?", "Paki-book po kami sa {item}.", "Pwede po bang ilipat ang booking namin sa {day}?",
              "Ika-cancel po namin ang reservation sa {day}.", "Bukas po ba kayo sa {day}? Pupunta sana kami.",
              "Kailangan po bang mag-reserve muna para sa {item}?", "Pa-reserve po para sa {n}, darating kami mga {time}.",
              "Pwede ba kaming sumali sa {item} sa {day}?"],
  "PRICE": ["Magkano po ang {item} per person?", "Magkano po lahat para sa {n} tao?", "May entrance fee po ba?",
            "Hm po ang {item}?", "Tumatanggap po ba kayo ng GCash?", "Kasama na po ba ang {item} sa bayad?",
            "May student discount po ba?", "May group rate po ba para sa {n}?", "Magkano po ang downpayment?",
            "Pwede po bang magbayad pagdating namin?"],
  "DIRECTIONS": ["Paano po pumunta sa farm galing {place}?", "Saan po mismo ang farm?", "Pwede po bang pa-send ng pin?",
                 "May parking po ba para sa van?", "Anong jeep po ang papunta sa farm galing {place}?", "Gaano po kalayo mula {place}?",
                 "Naliligaw po kami malapit sa {place}, saan po kami liliko?", "Kaya po ba ng tricycle hanggang farm?",
                 "Anong landmark po ang hahanapin namin?", "Ilang oras po ang biyahe galing {place}?"],
  "DIETARY_SAFETY": ["Allergic po sa {allergen} ang anak ko. Safe po ba ang {item}?", "May {allergen} po ba ang pagkain?",
                     "Bawal po ako sa {allergen}, pakisabi po sa cook.", "Vegetarian po kami, may pagkain po ba para sa amin?",
                     "Halal po ba ang tanghalian?", "Naka-wheelchair po ang nanay ko. Kaya po ba niya ang {item}?",
                     "Safe pa po ba ang trail kapag umuulan?", "May lilim po ba? Madaling mahilo sa init ang anak ko.",
                     "May bubuyog po ba o hayop na nangangagat? 3 years old po ang anak ko.", "Buntis po ako, nakakapagod po ba ang {item}?"],
  "OTHER": ["Hello po!", "Salamat po, ingat!", "Anong mga hayop po ang meron sa farm?", "Pwede po bang mag-picture?",
            "May wifi po ba?", "Kailan po ang best na panahon para bumisita?", "Ang saya po namin, salamat!", "May pasalubong po ba kayo?",
            "Magandang umaga po!", "Ano pong mga tanim ninyo?"]},
 "ko": {
  "BOOKING": ["{day}에 {n} {item} 예약할 수 있나요?", "{day} {time}에 예약하고 싶어요.", "{day}에 {n} 자리가 아직 있나요?",
              "{item} 예약 부탁드려요.", "예약을 {day}로 바꿀 수 있나요?", "{day} 예약을 취소하고 싶어요.",
              "{day}에 문 여나요? 방문하고 싶어요.", "{item}은 미리 예약해야 하나요?", "{n} 예약해 주세요. {time}쯤 도착해요.",
              "{day} 아침 {item}에 참여할 수 있나요?"],
  "PRICE": ["{item}은 1인당 얼마예요?", "{n}이면 가격이 어떻게 되나요?", "입장료가 있나요?", "{item} 얼마예요?",
            "카드 결제 되나요, 현금만 되나요?", "{item}도 가격에 포함인가요?", "학생 할인 있나요?", "{n} 단체 요금이 있나요?",
            "예약금은 얼마예요?", "도착해서 결제해도 되나요?"],
  "DIRECTIONS": ["{place}에서 농장까지 어떻게 가요?", "농장 정확한 위치가 어디예요?", "지도 위치 보내주실 수 있나요?",
                 "승합차 주차할 곳이 있나요?", "{place}에서 농장 가는 지프니가 뭐예요?", "{place}에서 얼마나 멀어요?",
                 "{place} 근처에서 길을 잃었어요. 어느 쪽이에요?", "트라이시클로 농장까지 갈 수 있나요?",
                 "어떤 건물을 보고 찾아가면 되나요?", "{place}에서 차로 얼마나 걸려요?"],
  "DIETARY_SAFETY": ["아들이 {allergen} 알레르기가 있어요. {item} 괜찮을까요?", "음식에 {allergen}이 들어가나요?",
                     "저는 {allergen}을 못 먹어요. 요리사분께 전해 주세요.", "저희는 채식주의자인데 먹을 음식이 있나요?",
                     "점심은 할랄인가요?", "어머니가 휠체어를 타세요. {item} 참여할 수 있나요?", "비가 오면 산책로가 안전한가요?",
                     "그늘이 있나요? 딸이 더위에 약해요.", "벌이나 무는 동물이 있나요? 아이가 세 살이에요.", "임신 중인데 {item}이 힘들까요?"],
  "OTHER": ["안녕하세요!", "정말 감사합니다, 또 봬요!", "농장에 어떤 동물이 있어요?", "사진 찍어도 되나요?", "와이파이 있나요?",
            "방문하기 가장 좋은 계절이 언제예요?", "정말 즐거웠어요, 감사합니다!", "기념품 파나요?", "좋은 아침이에요!", "어떤 작물을 키우세요?"]},
 "ja": {
  "BOOKING": ["{day}に{n}で{item}を予約できますか？", "{day}の{time}に予約したいです。", "{day}、{n}でまだ空きはありますか？",
              "{item}の予約をお願いします。", "予約を{day}に変更できますか？", "{day}の予約をキャンセルしたいです。",
              "{day}は営業していますか？伺いたいです。", "{item}は事前予約が必要ですか？", "{n}で予約お願いします。{time}ごろ着きます。",
              "{day}の朝の{item}に参加できますか？"],
  "PRICE": ["{item}は一人いくらですか？", "{n}だと料金はいくらですか？", "入場料はありますか？", "{item}はいくらですか？",
            "カードは使えますか？現金のみですか？", "{item}は料金に含まれていますか？", "学生割引はありますか？",
            "{n}の団体料金はありますか？", "予約金はいくらですか？", "着いてから支払ってもいいですか？"],
  "DIRECTIONS": ["{place}から農場までどう行けばいいですか？", "農場の正確な場所はどこですか？", "地図のピンを送ってもらえますか？",
                 "バンを停める駐車場はありますか？", "{place}から農場へ行くジプニーはどれですか？", "{place}からどのくらい遠いですか？",
                 "{place}の近くで迷いました。どちらですか？", "トライシクルで農場まで行けますか？", "目印になる建物は何ですか？",
                 "{place}から車でどのくらいかかりますか？"],
  "DIETARY_SAFETY": ["息子は{allergen}アレルギーです。{item}は大丈夫ですか？", "料理に{allergen}は入っていますか？",
                     "{allergen}が食べられません。料理人の方に伝えてください。", "ベジタリアンですが、食べられるものはありますか？",
                     "ランチはハラールですか？", "母は車椅子を使っています。{item}に参加できますか？", "雨の日でも道は安全ですか？",
                     "日陰はありますか？娘は暑さに弱いです。", "蜂やかむ動物はいますか？子どもは3歳です。", "妊娠中ですが、{item}は大変ですか？"],
  "OTHER": ["こんにちは！", "本当にありがとうございました！", "農場にはどんな動物がいますか？", "写真を撮ってもいいですか？",
            "Wi-Fiはありますか？", "訪れるのに一番いい季節はいつですか？", "とても楽しかったです、ありがとう！", "お土産は売っていますか？",
            "おはようございます！", "どんな作物を育てていますか？"]},
 "zh": {
  "BOOKING": ["{day}可以预约{n}的{item}吗？", "我想预约{day}{time}。", "{day}还有{n}的位置吗？", "请帮我们预约{item}。",
              "可以把预约改到{day}吗？", "我们想取消{day}的预约。", "{day}你们开门吗？我们想过去。", "{item}需要提前预约吗？",
              "请预约{n}，我们{time}左右到。", "{day}早上可以参加{item}吗？"],
  "PRICE": ["{item}每人多少钱？", "{n}一共多少钱？", "有门票吗？", "{item}多少钱？", "可以刷卡吗，还是只收现金？",
            "{item}包含在价格里吗？", "学生有优惠吗？", "{n}有团体价吗？", "订金是多少？", "到了再付款可以吗？"],
  "DIRECTIONS": ["从{place}怎么去你们的农场？", "农场的具体位置在哪里？", "可以发个定位给我吗？", "有可以停面包车的地方吗？",
                 "从{place}坐哪辆吉普尼去农场？", "离{place}有多远？", "我们在{place}附近迷路了，往哪边走？",
                 "三轮车能到农场吗？", "我们应该找什么地标？", "从{place}开车要多久？"],
  "DIETARY_SAFETY": ["我儿子对{allergen}过敏，{item}安全吗？", "食物里有{allergen}吗？", "我不能吃{allergen}，请告诉厨师。",
                     "我们吃素，有我们能吃的吗？", "午餐是清真的吗？", "我妈妈坐轮椅，她能参加{item}吗？", "下雨的话步道安全吗？",
                     "有遮阴的地方吗？我女儿怕热。", "有蜜蜂或会咬人的动物吗？孩子才三岁。", "我怀孕了，{item}会不会太累？"],
  "OTHER": ["你好！", "非常感谢，再见！", "农场里有什么动物？", "可以拍照吗？", "有无线网络吗？", "什么季节来最好？",
            "我们玩得很开心，谢谢！", "有卖纪念品吗？", "早上好！", "你们种什么作物？"]},
}
# Excluded on purpose: price questions about children ("Do kids get a discount?").
# They mention children, so by policy they are DIETARY_SAFETY or excluded; they are excluded.
# Casual, short, chat-style templates added after v1 missed lowercase Taglish and
# abbreviated English. Kept separate so reports can show their effect.
CASUAL = {
 "en": {"BOOKING": ["slots for {n} on {day}?", "can we come {day}", "book {item} pls", "still open {day}? {n} pax"],
        "PRICE": ["price for {item}?", "rates pls", "how much total for {n}", "entrance fee?"],
        "DIRECTIONS": ["address pls", "location?", "which way from {place}", "is it near {place}"],
        "OTHER": ["nice farm!", "ok thanks", "cool see u", "ty!"]},
 "tl": {"BOOKING": ["pede pa {day}? {n} kami", "pa book po {item}", "open po kayo {day}?", "reserve po {n} pax"],
        "PRICE": ["hm po {item}", "magkano po lahat", "rate po for {n}?", "magkano entrance"],
        "DIRECTIONS": ["san po location nyo", "pa pin naman po", "malapit lang po ba sa {place}", "pano po pumunta jan galing {place}"],
        "OTHER": ["sige po salamat", "ang ganda po!", "ok po noted", "thank u po"]},
 "ko": {"BOOKING": ["{day} {n} 되나요", "{item} 예약 가능?"], "PRICE": ["얼마예요", "{item} 가격?"],
        "DIRECTIONS": ["주소 알려주세요", "그랩으로 갈 수 있어요?"], "OTHER": ["감사해요", "좋아요!"]},
 "ja": {"BOOKING": ["{day}{n}空いてますか", "{item}予約したい"], "PRICE": ["いくら？", "{item}の値段は"],
        "DIRECTIONS": ["住所教えて", "グラブで行けますか"], "OTHER": ["ありがとう！", "いいね"]},
 "zh": {"BOOKING": ["{day}{n}有位吗", "想订{item}"], "PRICE": ["多少钱", "{item}价格？"],
        "DIRECTIONS": ["地址发我一下", "打Grab能到吗"], "OTHER": ["谢谢！", "好的"]},
}
for _l, _d in CASUAL.items():
    for _i, _ts in _d.items():
        T[_l][_i] = T[_l][_i] + _ts

INTENTS = ["BOOKING", "PRICE", "DIRECTIONS", "DIETARY_SAFETY", "OTHER"]
LANGS = list(T)


def templates():
    """Yield (template_id, lang, intent, template)."""
    for lang in LANGS:
        for intent in INTENTS:
            for i, t in enumerate(T[lang][intent]):
                yield f"{lang}-{intent}-{i:02d}", lang, intent, t


def fill(template: str, lang: str, rng) -> tuple[str, dict]:
    import string
    keys = [k for _, k, _, _ in string.Formatter().parse(template) if k]
    vals = {k: rng.choice(FILL[lang][k]) for k in keys}
    slots = {SLOT_OF[k]: v for k, v in vals.items()}
    return template.format(**vals), slots


def split_ids(test_frac=0.2, seed=13):
    """Template-level split, stratified by (lang, intent): about 2 of 10 templates go to test."""
    import random
    rng = random.Random(seed)
    test = set()
    for lang in LANGS:
        for intent in INTENTS:
            ids = [f"{lang}-{intent}-{i:02d}" for i in range(len(T[lang][intent]))]
            test.update(rng.sample(ids, max(1, round(len(ids) * test_frac))))
    return test


_EN_SUB = [("how much", ["hw much", "hm", "how much"]), ("please", ["pls", "plz"]), ("people", ["ppl", "pax"]),
           ("tomorrow", ["tmrw", "tom"]), (" you", [" u"]), (" for ", [" 4 ", " for "]), ("where", ["wer", "where"])]
_TL_SUB = [(" po", ["", " po"]), ("magkano", ["hm", "magkano", "mgkano"]), ("saan", ["san"]), ("pwede", ["pede", "pwde"]),
           ("paano", ["pano"]), ("ninyo", ["nyo"]), ("diyan", ["jan"]), ("kayo", ["kau", "kayo"])]


def messy(text: str, lang: str, rng) -> str:
    """Chat-style variant: lowercase, no final punctuation, common abbreviations."""
    t = text.rstrip("?!.。？！")
    if lang in ("en", "tl"):
        t = t.lower()
        for a, opts in (_EN_SUB if lang == "en" else _TL_SUB):
            if a in t and rng.random() < 0.6:
                t = t.replace(a, rng.choice(opts))
    elif lang == "ko" and rng.random() < 0.3:
        t = t.replace(" ", "")
    return t


def build(per_template=12, seed=13, messy_frac=0.5):
    """Return (train, test) lists of dicts: text, lang, intent, slots, template_id."""
    import random
    rng = random.Random(seed)
    test_ids = split_ids(seed=seed)
    train, test = [], []
    for tid, lang, intent, t in templates():
        seen = set()
        n = per_template if "{" in t else 2
        for _ in range(n * 3):
            text, slots = fill(t, lang, rng)
            if text in seen:
                continue
            seen.add(text)
            row = {"text": text, "lang": lang, "intent": intent, "slots": slots, "template_id": tid}
            if tid in test_ids:
                test.append(row)
            else:
                train.append(row)
                if rng.random() < messy_frac:   # augmentation on train only
                    m = messy(text, lang, rng)
                    if m != text and all(v.lower() in m.lower() for v in slots.values()):
                        train.append({**row, "text": m, "slots": {k: next(m[i:i + len(v)] for i in range(len(m))
                                      if m[i:i + len(v)].lower() == v.lower()) for k, v in slots.items()}})
            if len(seen) >= n:
                break
    return train, test


Overwriting templates.py


## 3. Build the Dataset

In [4]:
import json, random, re, tarfile, torch, os
from collections import Counter
from pathlib import Path
import templates as TP, bank as SB

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Turn on the GPU in Session options."
OUT = Path("/kaggle/working/intent_out"); OUT.mkdir(exist_ok=True)
random.seed(13)

train, test = TP.build(per_template=12)
print("templates train/test:", len(train), len(test))

templates train/test: 1310 250


### OTHER Negatives from MASSIVE

Only the eight guardrail-approved MASSIVE labels are used, and only as OTHER. A sentence is dropped if its English parallel contains a safety, price, booking, or route keyword, or if the safety detector fires on it. MASSIVE sentences are added to training only.

In [5]:
import urllib.request
URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
tgz = Path("/kaggle/working/massive.tar.gz")
if not tgz.exists():
    urllib.request.urlretrieve(URL, tgz)
with tarfile.open(tgz) as t:
    t.extractall("/kaggle/working/massive", filter="data")
D = Path("/kaggle/working/massive/1.1/data")
def read(loc): return [json.loads(l) for l in open(D / f"{loc}.jsonl", encoding="utf-8")]

OTHER_OK = {"general_greet", "general_joke", "datetime_query", "datetime_convert",
            "qa_definition", "qa_factoid", "news_query", "alarm_query"}
KEYWORDS = ["food", "eat", "allerg", "sick", "hospital", "doctor", "medicine", "child", "kid", "baby",
            "wheelchair", "disab", "pregnan", "storm", "typhoon", "flood", "rain", "heat", "weather",
            "earthquake", "price", "cost", "how much", "pay", "book", "reserv", "where is", "how do i get"]
en = {r["id"]: r for r in read("en-US")}
LOC = {"en": "en-US", "tl": "tl-PH", "ko": "ko-KR", "ja": "ja-JP", "zh": "zh-CN"}
PER_LANG = 60
massive = []
for lang, loc in LOC.items():
    pool = [r for r in read(loc) if r["partition"] == "train" and r["intent"] in OTHER_OK
            and not any(k in en[r["id"]]["utt"].lower() for k in KEYWORDS) and not SB.detect(r["utt"])]
    random.shuffle(pool)
    massive += [{"text": r["utt"], "lang": lang, "intent": "OTHER", "slots": {},
                 "template_id": f"massive-{r['intent']}"} for r in pool[:PER_LANG]]
train += massive
random.shuffle(train)
print("train:", len(train), Counter(r["intent"] for r in train))
print("test :", len(test), Counter(r["intent"] for r in test))

train: 1610 Counter({'BOOKING': 605, 'OTHER': 368, 'PRICE': 221, 'DIETARY_SAFETY': 211, 'DIRECTIONS': 205})
test : 250 Counter({'BOOKING': 109, 'DIETARY_SAFETY': 49, 'PRICE': 40, 'DIRECTIONS': 40, 'OTHER': 12})


### Wild Test Set

Template test sentences are still clean and template-shaped. These hand-written messages use Taglish, typos, missing punctuation, and mixed topics, which is closer to real chat. They are never used for training. They are a small sanity check, not a benchmark; real guest messages should replace them when available.

In [6]:
WILD = [
 ("hi po pwede pa ba mag book this sat? 5 kami", "tl", "BOOKING"),
 ("magkano po entrance tsaka yung lunch", "tl", "PRICE"),
 ("san po kayo banda? from cubao kami", "tl", "DIRECTIONS"),
 ("may allergy po si bunso sa hipon, ok lang ba ulam?", "tl", "DIETARY_SAFETY"),
 ("ang ganda po ng farm nyo! thank u", "tl", "OTHER"),
 ("hey can we come sunday arnd 10? 3 adults", "en", "BOOKING"),
 ("hw much 4 the strawberry thing", "en", "PRICE"),
 ("google maps says road closed?? which way now", "en", "DIRECTIONS"),
 ("my wife has celiac, anything gluten free for lunch?", "en", "DIETARY_SAFETY"),
 ("love the goats lol", "en", "OTHER"),
 ("토요일 4명 가능?", "ko", "BOOKING"),
 ("가격 알려주세요", "ko", "PRICE"),
 ("택시 기사님한테 뭐라고 말하면 돼요?", "ko", "DIRECTIONS"),
 ("애가 계란 먹으면 두드러기 나요", "ko", "DIETARY_SAFETY"),
 ("土曜4名いけますか", "ja", "BOOKING"),
 ("料金教えてください", "ja", "PRICE"),
 ("タクシーの運転手に何て言えばいい？", "ja", "DIRECTIONS"),
 ("娘がえびだめなんです", "ja", "DIETARY_SAFETY"),
 ("周六四个人可以吗", "zh", "BOOKING"),
 ("怎么收费", "zh", "PRICE"),
 ("司机找不到路，怎么走", "zh", "DIRECTIONS"),
 ("我老公花生过敏，很严重", "zh", "DIETARY_SAFETY"),
]
print(len(WILD), "wild messages")

22 wild messages


## 4. Prompt Format and Model

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM = ("You classify messages from guests of a Philippine farm tour. Reply with JSON only: "
          '{"intent": one of BOOKING, PRICE, DIRECTIONS, DIETARY_SAFETY, OTHER, "slots": {...}}. '
          "Slots may include party_size, date, time, place, item, allergen, copied exactly from the message. "
          "Anything about food, allergies, health, children, accessibility, or weather risk is DIETARY_SAFETY.")
tok = AutoTokenizer.from_pretrained(BASE)
def prompt(text):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}],
                                   tokenize=False, add_generation_prompt=True)
def target(r):
    return json.dumps({"intent": r["intent"], "slots": r["slots"]}, ensure_ascii=False) + tok.eos_token
def encode(r):
    p = tok(prompt(r["text"]), add_special_tokens=False)["input_ids"]
    a = tok(target(r), add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}   # loss on the answer only
enc = [encode(r) for r in train]
print("max length:", max(len(e["input_ids"]) for e in enc))
print(prompt(train[0]["text"]) + target(train[0]))

max length: 172
<|im_start|>system
You classify messages from guests of a Philippine farm tour. Reply with JSON only: {"intent": one of BOOKING, PRICE, DIRECTIONS, DIETARY_SAFETY, OTHER, "slots": {...}}. Slots may include party_size, date, time, place, item, allergen, copied exactly from the message. Anything about food, allergies, health, children, accessibility, or weather risk is DIETARY_SAFETY.<|im_end|>
<|im_start|>user
농장정확한위치가어디예요<|im_end|>
<|im_start|>assistant
{"intent": "DIRECTIONS", "slots": {}}<|im_end|>


## 5. Fine-Tune with LoRA

In [8]:
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(DEVICE)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                       target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id); lab = torch.full((len(batch), L), -100); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"]); ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

EPOCHS, BS, LR = 3, 16, 2e-4
dl = DataLoader(enc, batch_size=BS, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = EPOCHS * len(dl); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()
model.train(); step = 0
for ep in range(EPOCHS):
    tot = 0
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**b).loss
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        tot += loss.item(); step += 1
    print(f"epoch {ep + 1}: mean loss {tot / len(dl):.4f}")
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


/tmp/ipykernel_397/2493106477.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1: mean loss 0.0950
epoch 2: mean loss 0.0073
epoch 3: mean loss 0.0013


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
    

## 6. Evaluate

Each message is classified by the model. The final intent applies the safety override: if the rule detector fires, the intent becomes DIETARY_SAFETY. Recall on DIETARY_SAFETY is the most important number, because a missed safety message would reach free translation.

In [9]:
def predict(texts, bs=32):
    tok.padding_side = "left"; out = []
    for i in range(0, len(texts), bs):
        x = tok([prompt(t) for t in texts[i:i+bs]], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            y = model.generate(**x, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
        for s in tok.batch_decode(y[:, x["input_ids"].shape[1]:], skip_special_tokens=True):
            try:
                j = json.loads(s.strip()); out.append((j.get("intent", "INVALID"), j.get("slots", {}), True))
            except Exception:
                m = re.search(r"(BOOKING|PRICE|DIRECTIONS|DIETARY_SAFETY|OTHER)", s)
                out.append((m.group(1) if m else "INVALID", {}, False))
    return out

def report(rows, preds, name):
    gold = [r["intent"] for r in rows]
    raw = [p[0] for p in preds]
    final = [SB.route(r["text"], p) for r, p in zip(rows, raw)]
    res = {"n": len(rows), "valid_json": round(sum(p[2] for p in preds) / len(preds), 3)}
    for tag, pred in [("model_only", raw), ("with_safety_override", final)]:
        per = {}
        for c in TP.INTENTS:
            tp = sum(g == c and p == c for g, p in zip(gold, pred)); fp = sum(g != c and p == c for g, p in zip(gold, pred))
            fn = sum(g == c and p != c for g, p in zip(gold, pred))
            per[c] = {"precision": round(tp / (tp + fp), 3) if tp + fp else None, "recall": round(tp / (tp + fn), 3) if tp + fn else None}
        res[tag] = {"accuracy": round(sum(g == p for g, p in zip(gold, pred)) / len(gold), 3), "per_intent": per,
                    "safety_missed": [r["text"] for r, p in zip(rows, pred) if r["intent"] == "DIETARY_SAFETY" and p != "DIETARY_SAFETY"]}
    by_lang = {}
    for l in sorted({r["lang"] for r in rows}):
        idx = [i for i, r in enumerate(rows) if r["lang"] == l]
        by_lang[l] = round(sum(gold[i] == final[i] for i in idx) / len(idx), 3)
    res["accuracy_by_lang_with_override"] = by_lang
    if rows and "slots" in rows[0]:
        res["slot_exact_match"] = round(sum(p[1] == r["slots"] for r, p in zip(rows, preds)) / len(rows), 3)
    res["errors"] = [{"text": r["text"], "gold": g, "pred": p} for r, g, p in zip(rows, gold, final) if g != p][:40]
    print(f"== {name}: accuracy model {res['model_only']['accuracy']:.1%} | with override {res['with_safety_override']['accuracy']:.1%}"
          f" | DIETARY_SAFETY recall {res['model_only']['per_intent']['DIETARY_SAFETY']['recall']} -> "
          f"{res['with_safety_override']['per_intent']['DIETARY_SAFETY']['recall']}")
    print("   by language:", by_lang, "| valid JSON:", res["valid_json"], "| slots exact:", res.get("slot_exact_match"))
    return res

RESULTS = {"base": BASE, "train_size": len(train), "test_size": len(test)}
RESULTS["template_test"] = report(test, predict([r["text"] for r in test]), "template test (unseen templates)")
wild_rows = [{"text": t, "lang": l, "intent": i} for t, l, i in WILD]
RESULTS["wild_test"] = report(wild_rows, predict([r["text"] for r in wild_rows]), "wild test (hand-written)")
for e in RESULTS["wild_test"]["errors"]: print("   wild error:", e)
(OUT / "metrics.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2))

== template test (unseen templates): accuracy model 96.0% | with override 98.4% | DIETARY_SAFETY recall 0.878 -> 1.0
   by language: {'en': 1.0, 'ja': 0.976, 'ko': 1.0, 'tl': 0.962, 'zh': 1.0} | valid JSON: 1.0 | slots exact: 0.756
== wild test (hand-written): accuracy model 100.0% | with override 100.0% | DIETARY_SAFETY recall 1.0 -> 1.0
   by language: {'en': 1.0, 'ja': 1.0, 'ko': 1.0, 'tl': 1.0, 'zh': 1.0} | valid JSON: 1.0 | slots exact: None


3430

## 7. Try Your Own Messages

In [10]:
for t in ["Pwede po ba kami pumunta bukas ng hapon, 7 kami?", "我女儿对牛奶过敏", "How much is parking?", "Salamat po sa masarap na lunch!"]:
    intent, slots, _ = predict([t])[0]
    print(f"{t}\n   model: {intent} {slots} | final: {SB.route(t, intent)}")

Pwede po ba kami pumunta bukas ng hapon, 7 kami?
   model: BOOKING {'date': 'hapon', 'party_size': '7'} | final: BOOKING
我女儿对牛奶过敏
   model: DIETARY_SAFETY {'allergen': '牛奶'} | final: DIETARY_SAFETY
How much is parking?
   model: PRICE {} | final: PRICE
Salamat po sa masarap na lunch!
   model: PRICE {'item': 'lunch'} | final: PRICE


## 8. Merge and Export to GGUF

The LoRA weights are merged into the base model, then converted with llama.cpp's converter to GGUF at q8_0 (about 0.5 GB), which runs on phones through llama.cpp bindings. q8_0 is used because it needs no compiled tools; a smaller Q4_K_M file can be made later with `llama-quantize`.

In [11]:
merged = model.merge_and_unload().to(torch.float16)
merged.save_pretrained("/kaggle/working/merged"); tok.save_pretrained("/kaggle/working/merged")
del model, merged; torch.cuda.empty_cache()
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!PYTHONPATH=/kaggle/working/llama.cpp/gguf-py python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outfile /kaggle/working/intent_out/turaista-intent-v2-q8_0.gguf --outtype q8_0
!ls -lh /kaggle/working/intent_out

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

fatal: destination path '/kaggle/working/llama.cpp' already exists and is not an empty directory.
INFO:hf-to-gguf:Loading model: merged
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {128}
INF

## 9. Package

`intent_out.zip` contains the GGUF model, `metrics.json`, and the app prompt. The app must use exactly the same system prompt and chat template as training.

In [12]:
(OUT / "app_prompt.json").write_text(json.dumps({"system": SYSTEM, "chat_template": "qwen2.5 (from tokenizer)",
    "max_new_tokens": 80, "temperature": 0, "post_rule": "if safety detector fires, intent = DIETARY_SAFETY"}, indent=2))
import shutil
from IPython.display import FileLink
shutil.make_archive("/kaggle/working/intent_out", "zip", OUT)
os.chdir("/kaggle/working"); FileLink("intent_out.zip")

/kaggle/working/intent_out.zip